# Customer Single View & Designing Customer Data Platform

สร้าง Customer Single View (CSV) จากข้อมูลหลายแหล่ง เพื่อเตรียมข้อมูลสำหรับ Customer Analytics

## 1. โหลดข้อมูล

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
customers    = pd.read_csv('../data/sample/customers.csv', parse_dates=['registration_date'])
transactions = pd.read_csv('../data/sample/transactions.csv', parse_dates=['transaction_date'])
products     = pd.read_csv('../data/sample/products.csv')

print(f'Customers:     {customers.shape}')
print(f'Transactions:  {transactions.shape}')
print(f'Products:      {products.shape}')

In [ ]:
customers.head()

In [ ]:
transactions.head()

In [ ]:
products.head()

## 2. Data Quality Check

In [ ]:
print('=== Customers ===')
print(customers.isnull().sum())
print('\n=== Transactions ===')
print(transactions.isnull().sum())
print('\n=== Products ===')
print(products.isnull().sum())

In [ ]:
print(f'Unique customers in transactions: {transactions["customer_id"].nunique()}')
print(f'Customers without transactions: {len(set(customers["customer_id"]) - set(transactions["customer_id"]))}')

## 3. Feature Engineering — Customer Single View

In [ ]:
# — Aggregate transaction features —
avg = transactions.groupby('customer_id').agg(
    recency=('transaction_date', lambda x: (x.max() - x.min()).days),
    frequency=('transaction_id', 'count'),
    monetary=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max'),
    first_txn=('transaction_date', 'min'),
    last_txn=('transaction_date', 'max'),
).reset_index()
avg['recency'] = avg['recency'].clip(lower=0)
print(avg.head())

In [ ]:
# — Channel preference —
channel_pivot = transactions.pivot_table(
    index='customer_id', columns='channel', values='amount', aggfunc='count', fill_value=0
).add_prefix('channel_')
channel_pivot['preferred_channel'] = channel_pivot.idxmax(axis=1)
channel_pivot.head()

In [ ]:
csv = customers.merge(avg, on='customer_id', how='left').merge(channel_pivot, on='customer_id', how='left')
csv.fillna({'frequency':0,'monetary':0,'avg_amount':0,'max_amount':0}, inplace=True)
print(f'CSV shape: {csv.shape}')
csv.head()

## 4. Visualize CSV Features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
cols = ['age','income','frequency','monetary','avg_amount','recency']
for ax, col in zip(axes.flat, cols):
    sns.histplot(csv[col], bins=40, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.savefig('../data/sample/csv_distributions.png', dpi=100)
plt.show()

## 5. สรุป

✅ Customer Single View สร้างสำเร็จ — มีข้อมูล 500 customers พร้อม 12 features พร้อมนำไปใช้วิเคราะห์ต่อ (CLV, Churn, Segmentation)